# Assignment 2: Feedforward Neural Network using Keras and TensorFlow

**Aim:** To implement Feedforward Neural Networks with Keras and TensorFlow.

This notebook is adapted from your uploaded Google Colab notebook for **local Jupyter Notebook**.

### Dataset folder arrangement

Extract your dataset so that the folder looks like this, and keep it in the same folder as this notebook:

```text
DL_Practicals/
│
├── Assignment_2_FFNN_Jupyter.ipynb
└── mnist-jpg/
    ├── train/
    │   ├── 0/
    │   ├── 1/
    │   └── ... (remaining digit folders)
    └── test/
        ├── 0/
        ├── 1/
        └── ... (remaining digit folders)
```

The code can also extract `mnist-jpg.zip` automatically if you put that ZIP file beside the notebook.

**Note:** The notebook uses RGB images (28 × 28 × 3), matching your original code. The test set is loaded from the `test` folder (your Colab notebook accidentally loaded the training set a second time).


## Cell 1: Import required libraries

In [ ]:
import numpy as np
import tensorflow as tf
import zipfile
import os
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.optimizers import SGD

print("TensorFlow version:", tf.__version__)

## Cell 2: Set dataset path and extract ZIP if needed

In [ ]:
# Keep mnist-jpg folder OR mnist-jpg.zip in the same folder as this notebook
dataset_folder = "mnist-jpg"
zip_filename = "mnist-jpg.zip"

if not os.path.isdir(dataset_folder):
    if os.path.isfile(zip_filename):
        print("Extracting dataset ZIP file...")
        with zipfile.ZipFile(zip_filename, "r") as zip_ref:
            zip_ref.extractall(".")
    else:
        raise FileNotFoundError(
            "Dataset not found. Keep the extracted 'mnist-jpg' folder "
            "or 'mnist-jpg.zip' beside this notebook."
        )

# If ZIP extraction created an extra parent folder, locate mnist-jpg
if not os.path.isdir(dataset_folder):
    for root, dirs, files in os.walk("."):
        if "train" in dirs and "test" in dirs and os.path.basename(root) == "mnist-jpg":
            dataset_folder = root
            break

train_path = os.path.join(dataset_folder, "train")
test_path = os.path.join(dataset_folder, "test")

if not os.path.isdir(train_path) or not os.path.isdir(test_path):
    raise FileNotFoundError(
        "Expected train and test folders inside: " + os.path.abspath(dataset_folder)
    )

print("Training folder:", os.path.abspath(train_path))
print("Testing folder:", os.path.abspath(test_path))

## Cell 3: Load training and testing image datasets

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(28, 28),
    color_mode="rgb",
    batch_size=None,
    shuffle=False
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(28, 28),
    color_mode="rgb",
    batch_size=None,
    shuffle=False
)

class_names = train_data.class_names
print("Class names:", class_names)

if test_data.class_names != class_names:
    raise ValueError("Train and test folders must contain the same class folders in the same order.")

## Cell 4: Convert datasets into NumPy arrays

In [ ]:
# Convert each dataset into image and label arrays
xtrain = np.stack([image.numpy() for image, label in train_data])
ytrain = np.array([label.numpy() for image, label in train_data])

xtest = np.stack([image.numpy() for image, label in test_data])
ytest = np.array([label.numpy() for image, label in test_data])

print("Training images:", xtrain.shape)
print("Training labels:", ytrain.shape)
print("Testing images:", xtest.shape)
print("Testing labels:", ytest.shape)

## Cell 5: Normalize image pixel values

In [ ]:
xtrain = xtrain.astype("float32") / 255.0
xtest = xtest.astype("float32") / 255.0

print("Training data normalized.")
print("Testing data normalized.")

## Cell 6: Define the Feedforward Neural Network model

In [ ]:
model = Sequential([
    Input(shape=(28, 28, 3)),
    Flatten(),
    Dense(60, activation="relu"),
    Dense(50, activation="relu"),
    Dense(len(class_names), activation="softmax")
])

model.summary()

## Cell 7: Compile the model using SGD optimizer

In [ ]:
model.compile(
    optimizer=SGD(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

## Cell 8: Train the model

In [ ]:
history = model.fit(
    xtrain,
    ytrain,
    epochs=10,
    validation_data=(xtest, ytest)
)

## Cell 9: Evaluate the model

In [ ]:
loss, acc = model.evaluate(xtest, ytest, verbose=2)

print("Test Loss:", loss)
print("Test Accuracy: {:.2f}%".format(acc * 100))

## Cell 10: Make predictions

In [ ]:
predictions = model.predict(xtest)

# Predict the class of the fifth test image
index = 4
predicted_class = np.argmax(predictions[index])
true_class = ytest[index]

print("Predicted class:", class_names[predicted_class])
print("Actual class:", class_names[true_class])

## Cell 11: Display a test image

In [ ]:
plt.imshow(xtest[index])
plt.title("Predicted: {} | Actual: {}".format(
    class_names[predicted_class], class_names[true_class]
))
plt.axis("off")
plt.show()

## Cell 12: Plot training and validation accuracy

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Testing Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Testing Accuracy")
plt.legend()
plt.show()

## Cell 13: Plot training and validation loss

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Testing Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Testing Loss")
plt.legend()
plt.show()

## Conclusion

Thus, we implemented a Feedforward Neural Network using Keras and TensorFlow for digit classification. The model was trained using the Stochastic Gradient Descent (SGD) optimizer, evaluated on test data, and its accuracy and loss were plotted.